# 模块概述

TraderDumper 是 WonderTrader 交易数据转储模块，负责从交易接口获取交易数据（账户、订单、成交、持仓）并转储到外部系统。主要包括：
- 交易适配器管理和生命周期控制
- 交易接口的动态加载和适配
- 交易数据的标准化和转发
- C语言导出接口，支持跨语言调用
- 支持一次性查询和持续刷新两种模式

1. **接口层**（TraderDumper.h/cpp）：
   - 提供C语言导出接口，支持Python、C#等外部语言调用
   - 封装Dumper类的功能，提供简单的生命周期管理
   - 使用extern "C"确保C++代码可以被C语言调用
   - 提供完整的初始化、配置、运行、释放流程

2. **核心管理层**（Dumper.h/cpp）：
   - Dumper：交易数据转储核心类，管理整个转储流程
   - 管理TraderAdapterMgr，控制所有交易通道的生命周期
   - 保存外部注册的回调函数，用于转发交易数据
   - 支持一次性查询模式和持续刷新模式
   - 在持续刷新模式下，启动后台线程定时刷新数据

3. **适配器层**（TraderAdapter.h/cpp）：
   - TraderAdapter：交易适配器类，封装单个交易通道的接口调用
   - TraderAdapterMgr：交易适配器管理器，管理多个交易通道适配器
   - 动态加载交易模块DLL/so（如TraderCTP、TraderXTP等）
   - 实现ITraderSpi接口，接收交易接口的回调通知
   - 将交易接口返回的数据转换为标准格式

4. **定义层**（PorterDefs.h）：
   - 定义回调函数类型：FuncOnAccount、FuncOnOrder、FuncOnTrade、FuncOnPosition
   - 定义数据结构，确保跨平台兼容性
   - 使用PORTER_FLAG导出标志，确保跨DLL/so调用的兼容性

5. **工具支持层**（WtHelper.h/cpp）：
   - WtHelper：辅助工具类，提供路径管理和模块目录管理功能
   - 支持Windows和Unix平台的路径操作
   - 提供当前工作目录和模块目录的获取和设置功能

# 层次关系图
```mermaid
graph LR
    %% 样式定义
    classDef interfaceClass fill:#e1f5ff,stroke:#01579b,stroke-width:3px,color:#000;
    classDef coreClass fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef adapterClass fill:#f3e5f5,stroke:#4a148c,stroke-width:2px,color:#000;
    classDef mgrClass fill:#ffe0b2,stroke:#e65100,stroke-width:2px,color:#000;
    classDef defClass fill:#fffde7,stroke:#f57f17,stroke-width:2px,color:#000;
    classDef utilClass fill:#fce4ec,stroke:#880e4f,stroke-width:2px,color:#000;
    classDef externalClass fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;

    %% 外部接口层
    subgraph ExternalLayer["外部接口层 - C语言导出接口"]
        direction TB
        TraderDumperC["TraderDumper.h/cpp<br/>C语言导出接口<br/>• register_callbacks<br/>• init<br/>• config<br/>• run<br/>• release"]:::interfaceClass
    end

    %% 定义层
    subgraph DefsLayer["定义层 - 回调函数类型"]
        direction TB
        PorterDefs["PorterDefs.h<br/>回调函数类型定义<br/>• FuncOnAccount<br/>• FuncOnOrder<br/>• FuncOnTrade<br/>• FuncOnPosition"]:::defClass
    end

    %% 核心管理层
    subgraph CoreLayer["核心管理层 - 数据转储控制"]
        direction TB
        Dumper["Dumper<br/>数据转储核心类<br/>• 管理TraderAdapterMgr<br/>• 保存回调函数<br/>• 处理数据转发<br/>• 后台刷新线程"]:::coreClass
    end

    %% 适配器层
    subgraph AdapterLayer["适配器层 - 交易接口适配"]
        direction TB
        TraderAdapterMgr["TraderAdapterMgr<br/>交易适配器管理器<br/>• 管理多个适配器<br/>• 统一启动/停止<br/>• 统一刷新接口"]:::mgrClass
        TraderAdapter["TraderAdapter<br/>交易适配器<br/>• 动态加载交易模块<br/>• 实现ITraderSpi接口<br/>• 数据格式转换<br/>• 连接和查询管理"]:::adapterClass
    end

    %% 工具支持层
    subgraph UtilLayer["工具支持层 - 辅助功能"]
        direction TB
        WtHelper["WtHelper<br/>辅助工具类<br/>• 路径管理<br/>• 模块目录管理<br/>• 跨平台支持"]:::utilClass
    end

    %% 外部系统
    subgraph ExternalSys["外部系统"]
        direction TB
        ExternalCallbacks["外部回调函数<br/>• Python/C#等调用<br/>• 接收交易数据"]:::externalClass
        TraderModules["交易模块DLL/so<br/>• TraderCTP<br/>• TraderXTP<br/>• TraderOES等"]:::externalClass
    end

    %% 继承和实现关系
    TraderAdapter -.->|"实现"| ITraderSpi
    Dumper -.->|"使用回调"| PorterDefs

    %% 组合关系
    TraderDumperC -->|"委托给"| Dumper
    Dumper -->|"管理"| TraderAdapterMgr
    TraderAdapterMgr -->|"管理"| TraderAdapter
    Dumper -->|"使用"| PorterDefs
    TraderAdapter -->|"使用"| WtHelper
    Dumper -->|"使用"| WtHelper

    %% 数据流关系
    TraderModules -->|"回调"| TraderAdapter
    TraderAdapter -->|"转发数据"| Dumper
    Dumper -->|"调用回调"| ExternalCallbacks

    %% 配置和初始化关系
    TraderDumperC -->|"注册回调"| Dumper
    TraderDumperC -->|"配置"| Dumper
    Dumper -->|"初始化"| TraderAdapterMgr
    TraderAdapterMgr -->|"加载模块"| TraderModules

    %% 应用样式
    class TraderDumperC interfaceClass
    class Dumper coreClass
    class TraderAdapterMgr,TraderAdapter adapterClass
    class TraderAdapterMgr mgrClass
    class PorterDefs defClass
    class WtHelper utilClass
    class ExternalCallbacks,TraderModules externalClass
```

# 基本类型定义 PorterDefs.h

# C语言导出接口 TraderDumper.h/cpp
定义了 TraderDumper 模块对外提供的C语言接口函数。这些函数可以被外部程序（如Python、C#等）通过DLL/so动态库方式调用。

## 获取全局Dumper单例实例 getDumper
```cpp
/**
 * @brief 获取全局Dumper单例实例
 * @return 返回Dumper类的引用
 */
Dumper& getDumper()
{
	static Dumper dumper; // 静态局部变量：全局唯一的Dumper实例，首次调用时初始化
	return dumper; // 返回Dumper实例的引用
}
```

## 注册数据回调函数 register_callbacks
```cpp
/**
 * @brief 注册数据回调函数的C接口实现
 * @param cbAccount 账户资金信息回调函数指针
 * @param cbOrder 订单信息回调函数指针
 * @param cbTrade 成交信息回调函数指针
 * @param cbPosition 持仓信息回调函数指针
 * 
 * 将外部传入的四个回调函数注册到Dumper实例中。
 * 当交易数据更新时，Dumper会调用这些回调函数通知外部。
 */
void register_callbacks(FuncOnAccount cbAccount, FuncOnOrder  cbOrder, FuncOnTrade cbTrade, FuncOnPosition cbPosition)
{
	getDumper().register_callbacks(cbAccount, cbOrder, cbTrade, cbPosition);  // 调用Dumper的注册回调方法
}
```

## 初始化日志系统 init
```cpp
/**
 * @brief 初始化日志系统的C接口实现
 * @param logProfile 日志配置文件路径
 * 
 * 初始化Dumper的日志系统，加载日志配置。
 * 必须在config之前调用。
 */
void init(const char* logProfile)
{
	getDumper().init(logProfile); // 调用Dumper的初始化方法，传入日志配置文件名
}
```

## 加载配置文件并初始化交易通道 config
```cpp
/**
 * @brief 加载配置文件的C接口实现
 * @param cfgfile 配置文件路径或配置内容
 * @param isFile 是否为文件路径
 * @return 返回配置是否成功
 * 
 * 配置Dumper，加载交易通道配置和基础数据文件。
 * 同时传入模块目录路径，用于定位交易模块DLL/so文件。
 */
bool config(const char* cfgfile, bool isFile)
{
	return getDumper().config(cfgfile, isFile, getBinDir()); // 调用Dumper的配置方法，传入配置文件和模块目录路径
}
```

## 启动数据转储 run
```cpp
/**
 * @brief 启动数据转储的C接口实现
 * @param bOnce 是否只运行一次
 * 
 * 启动Dumper运行，开始连接交易通道并查询数据。
 * 如果bOnce为true，会阻塞直到所有数据查询完成。
 */
void run(bool bOnce)
{
	getDumper().run(bOnce); // 调用Dumper的运行方法
}
```

## 释放资源并清理连接 release
```cpp
/**
 * @brief 释放资源的C接口实现
 * 
 * 释放Dumper的所有资源，断开交易连接，停止后台线程。
 * 应该在程序退出前调用。
 */
void release()
{
	getDumper().release(); // 调用Dumper的释放方法
}
```

# 数据转储核心 Dumper.h/cpp
```cpp
class Dumper
```

## 成员
- `FuncOnAccount _cb_account`：账户资金信息回调函数指针
- `FuncOnOrder _cb_order`：订单信息回调函数指针
- `FuncOnTrade _cb_trade`：成交信息回调函数指针
- `FuncOnPosition _cb_position`：持仓信息回调函数指针
- `StdThreadPtr	_worker`：后台工作线程指针，用于定时刷新数据（持续模式）
- `uint32_t _refresh_span`：刷新间隔（秒），定时刷新数据的时间间隔
- `bool _stopped`：停止标志（布尔值），用于控制后台线程的停止

## 注册数据回调函数 register_callbacks

## 初始化日志系统 init
```cpp
/**
 * @brief 初始化日志系统的实现
 * @param logProfile 日志配置文件路径
 * 
 * 初始化WonderTrader的日志系统，加载日志配置。
 */
void Dumper::init(const char* logProfile)
{
	WTSLogger::init(logProfile); // 调用WonderTrader日志系统的初始化方法，传入日志配置文件名
}
```

## 加载配置文件并初始化交易通道 config
```cpp
```

## 启动数据转储 run
```cpp
/**
 * @brief 启动数据转储的实现
 * @param bOnce 是否只运行一次，默认true
 * 
 * 启动流程：
 * - 启动所有交易适配器
 * - 如果bOnce为true，阻塞等待所有数据查询完成
 * - 如果bOnce为false，启动后台线程定时刷新数据
 */
void Dumper::run(bool bOnce /* = true */)
{
	g_adapterMgr.run();              // 启动所有交易适配器，开始连接和查询数据

	if(bOnce)                        // 如果是一次性模式
	{
		for (;;)                     // 无限循环
		{
			if (g_adapterMgr.isAnyAlive())  // 如果还有活跃的适配器在工作
				std::this_thread::sleep_for(std::chrono::seconds(1));  // 等待1秒
			else                     // 如果所有适配器都已完成
				break;               // 退出循环
		}
	}
	else                             // 如果是持续模式
	{
		_worker.reset(new StdThread([this]() {  // 创建后台工作线程
			
			while(!_stopped)         // 如果未停止
			{
				std::this_thread::sleep_for(std::chrono::seconds(_refresh_span));  // 等待刷新间隔时间
				g_adapterMgr.refresh();  // 刷新所有已完成的适配器数据
			}
			
		}));
	}
}
```

## 释放资源并清理连接 release
```cpp
/**
 * @brief 释放资源并清理连接的实现
 * 
 * 释放流程：
 * - 设置停止标志，停止后台刷新线程
 * - 等待后台线程结束
 * - 释放所有交易适配器资源（在适配器管理器的release中完成）
 */
void Dumper::release()
{
	_stopped = true;                  // 设置停止标志，通知后台线程停止
	if (_worker)                     // 如果后台线程存在
		_worker->join();              // 等待后台线程结束
}
```

## 处理账户资金数据 on_account
```cpp
/**
 * @brief 处理账户资金数据的实现
 * @param channelid 交易通道ID
 * @param curTDate 当前交易日
 * @param currency 货币类型
 * @param prebalance 上日余额
 * @param balance 当前余额
 * @param dynbalance 动态权益
 * @param closeprofit 平仓盈亏
 * @param dynprofit 浮动盈亏
 * @param fee 手续费
 * @param margin 占用保证金
 * @param deposit 入金
 * @param withdraw 出金
 * @param isLast 是否为最后一条数据
 * 
 * 接收TraderAdapter传来的账户数据，如果回调函数已注册，则转发到外部回调函数。
 */
void Dumper::on_account(const char* channelid, uint32_t curTDate, const char* currency, double prebalance, 
		double balance, double dynbalance, double closeprofit, double dynprofit, double fee, 
		double margin, double deposit, double withdraw, bool isLast)
{
	if (_cb_account)                 // 如果账户回调函数已注册
		_cb_account(channelid, curTDate, currency, prebalance, balance, dynbalance, closeprofit, dynprofit, fee, margin, deposit, withdraw, isLast);  // 调用外部回调函数
}
```

## 处理订单数据 on_order
```cpp
/**
 * @brief 处理订单数据的实现
 * @param channelid 交易通道ID
 * @param exchg 交易所代码
 * @param code 合约代码
 * @param curTDate 当前交易日
 * @param orderid 订单号
 * @param direct 买卖方向
 * @param offset 开平标志
 * @param volume 委托数量
 * @param leftover 剩余数量
 * @param traded 已成交数量
 * @param price 委托价格
 * @param ordertype 订单类型
 * @param pricetype 价格类型
 * @param ordertime 委托时间
 * @param state 订单状态
 * @param statemsg 状态信息
 * @param isLast 是否为最后一条数据
 * 
 * 接收TraderAdapter传来的订单数据，如果回调函数已注册，则转发到外部回调函数。
 */
void Dumper::on_order(const char* channelid, const char* exchg, const char* code, uint32_t curTDate, const char* orderid, uint32_t direct, uint32_t offset, double volume, double leftover, double traded, double price, uint32_t ordertype, uint32_t pricetype, WtUInt64 ordertime, uint32_t state, const char* statemsg, bool isLast)
{
	if (_cb_order)                   // 如果订单回调函数已注册
		_cb_order(channelid, exchg, code, curTDate, orderid, direct, offset, volume, leftover, traded, price, ordertype, pricetype, ordertime, state, statemsg, isLast);  // 调用外部回调函数
}
```

## 处理成交数据 on_trade
```cpp
/**
 * @brief 处理成交数据的实现
 * @param channelid 交易通道ID
 * @param exchg 交易所代码
 * @param code 合约代码
 * @param curTDate 当前交易日
 * @param tradeid 成交编号
 * @param orderid 订单号
 * @param direct 买卖方向
 * @param offset 开平标志
 * @param volume 成交数量
 * @param price 成交价格
 * @param amount 成交金额
 * @param ordertype 订单类型
 * @param tradetype 成交类型
 * @param tradetime 成交时间
 * @param isLast 是否为最后一条数据
 * 
 * 接收TraderAdapter传来的成交数据，如果回调函数已注册，则转发到外部回调函数。
 */
void Dumper::on_trade(const char* channelid, const char* exchg, const char* code, uint32_t curTDate, const char* tradeid, const char* orderid, 
		uint32_t direct, uint32_t offset, double volume, double price, double amount, uint32_t ordertype, uint32_t tradetype, WtUInt64 tradetime, bool isLast)
{
	if (_cb_trade)                   // 如果成交回调函数已注册
		_cb_trade(channelid, exchg, code, curTDate, tradeid, orderid, direct, offset, volume, price, amount, ordertype, tradetype, tradetime, isLast);  // 调用外部回调函数
}
```

## 处理持仓数据 on_position
```cpp
/**
 * @brief 处理持仓数据的实现
 * @param channelid 交易通道ID
 * @param exchg 交易所代码
 * @param code 合约代码
 * @param curTDate 当前交易日
 * @param direct 持仓方向
 * @param volume 持仓数量
 * @param cost 持仓成本
 * @param margin 占用保证金
 * @param avgpx 持仓均价
 * @param dynprofit 浮动盈亏
 * @param volscale 数量乘数
 * @param isLast 是否为最后一条数据
 * 
 * 接收TraderAdapter传来的持仓数据，如果回调函数已注册，则转发到外部回调函数。
 */
void Dumper::on_position(const char* channelid, const char* exchg, const char* code, uint32_t curTDate, uint32_t direct,
		double volume, double cost, double margin, double avgpx, double dynprofit, uint32_t volscale, bool isLast)
{
	if (_cb_position)                // 如果持仓回调函数已注册
		_cb_position(channelid, exchg, code, curTDate, direct, volume, cost, margin, avgpx, dynprofit, volscale, isLast);  // 调用外部回调函数
}
```

# 交易适配器层 TraderAdapter.h/cpp

## 交易适配器类 TraderAdapter
```cpp
class TraderAdapter : public ITraderSpi
```

### 成员
- `TraderAdapterMgr* _mgr`：交易适配器管理器指针，用于与管理器交互
- `WTSVariant* _cfg`：配置参数指针，保存交易通道的配置信息
- `std::string _id`：交易通道ID字符串，用于标识该通道
- `ITraderApi* _trader_api`：交易接口指针，指向动态加载的交易接口实例
- `FuncDeleteTrader	_remover`：交易接口删除函数指针，用于释放交易接口实例
- `IBaseDataMgr* _bd_mgr`：基础数据管理器指针，用于获取合约、品种等信息
- `uint32_t _date`：当前交易日（32位无符号整数），格式为YYYYMMDD，登录成功后设置
- `bool _done`：数据查询完成标志（布尔值），true表示已完成所有数据查询

### 核心属性

#### 获取交易通道ID id
```cpp
/**
 * @brief 获取交易通道ID
 * @return 返回通道ID的C风格字符串指针
 */
inline const char* id() const{ return _id.c_str(); }
```

#### 检查数据查询是否完成 isDone
```cpp
/**
 * @brief 检查数据查询是否完成
 * @return 返回是否已完成（布尔值），true表示已完成所有数据查询
 * 
 * 用于判断该交易通道是否已完成账户、持仓、订单、成交的查询。
 */
bool isDone() const { return _done; }
```

### 生命周期管理

#### 初始化交易适配器 init
```cpp
/**
 * @brief 初始化交易适配器的实现
 * @param id 交易通道ID
 * @param params 配置参数
 * @param bdMgr 基础数据管理器
 * @return 返回初始化是否成功
 * 
 * 初始化流程：
 * 1. 参数验证：检查params和_cfg是否有效
 * 2. 保存参数：保存通道ID、配置参数、基础数据管理器
 * 3. 加载交易模块：根据配置中的模块名，动态加载DLL/so
 * 4. 创建交易接口：调用模块的createTrader函数创建接口实例
 * 5. 初始化交易接口：调用接口的init方法，传入配置参数
 */
bool TraderAdapter::init(const char* id, WTSVariant* params, IBaseDataMgr* bdMgr)
```

#### 启动交易适配器 run
```cpp
/**
 * @brief 启动交易适配器的实现
 * @return 返回启动是否成功
 * 
 * 启动流程：
 * 1. 检查交易接口是否已创建
 * 2. 注册回调接口（this），使交易接口可以回调本类的方法
 * 3. 连接交易服务器，连接成功后会触发handleEvent回调
 */
bool TraderAdapter::run()
{
	if (_trader_api == NULL) // 如果交易接口未创建，启动失败
		return false;
	_trader_api->registerSpi(this); // 注册回调接口，this指针指向本TraderAdapter实例
	_trader_api->connect(); // 连接交易服务器，异步操作，连接结果通过handleEvent回调通知
	return true;
}
```

#### 释放资源 release
```cpp
/**
 * @brief 释放资源的实现
 * 
 * 释放流程：
 * 1. 注销回调接口（设置为NULL）
 * 2. 调用交易接口的release方法释放资源
 */
void TraderAdapter::release()
{
	if (_trader_api) // 如果交易接口存在
	{
		_trader_api->registerSpi(NULL); // 注销回调接口，防止回调已释放的对象
		_trader_api->release(); // 释放交易接口资源
	}
}
```

### 数据查询接口

#### 查询账户资金 queryFund
```cpp
/**
 * @brief 查询账户资金的实现
 * 
 * 主动查询账户资金信息，查询结果会通过onRspAccount回调返回。
 * 只有在交易日已确定（_date不为0）时才会执行查询。
 */
void TraderAdapter::queryFund()
{
	if (_date == 0)                 // 如果交易日未确定，不执行查询
		return;

	_trader_api->queryAccount();    // 调用交易接口查询账户资金
}
```

#### 查询持仓 queryPosition
```cpp
/**
 * @brief 查询持仓的实现
 * 
 * 主动查询持仓信息，查询结果会通过onRspPosition回调返回。
 * 只有在交易日已确定（_date不为0）时才会执行查询。
 */
void TraderAdapter::queryPosition()
{
	if (_date == 0)                 // 如果交易日未确定，不执行查询
		return;

	_trader_api->queryPositions();  // 调用交易接口查询持仓
}
```

### ITraderSpi 接口实现—事件回调

#### 处理交易事件 handleEvent
```cpp
/**
 * @brief 处理交易事件的实现
 * @param e 交易事件类型
 * @param ec 事件代码
 * 
 * 处理交易接口的事件通知：
 * - WTE_Connect: 连接事件，连接成功则自动登录，失败则标记为完成
 * - WTE_Close: 连接关闭事件，记录日志
 */
void TraderAdapter::handleEvent(WTSTraderEvent e, int32_t ec)
{
	if(e == WTE_Connect)            // 如果是连接事件
	{
		if(ec == 0)                  // 如果连接成功（错误码为0）
		{
			_trader_api->login(_cfg->getCString("user"), _cfg->getCString("pass"), _cfg->getCString("product"));  // 自动登录，传入用户名、密码、产品信息
		}
		else                         // 如果连接失败
		{
			WTSLogger::error("[{}]交易账号连接失败: {}", _id.c_str(), ec);  // 记录错误日志
			_mgr->decAlive();         // 通知管理器减少活跃计数
			_done = true;             // 标记为完成状态（失败也算完成）
		}
	}
	else if(e == WTE_Close)         // 如果是连接关闭事件
	{
		WTSLogger::error("[{}]交易账号连接已断开: {}", _id.c_str(), ec);  // 记录错误日志
	}
}
```

#### 登录结果回调 onLoginResult
```cpp
/**
 * @brief 登录结果回调的实现
 * @param bSucc 登录是否成功
 * @param msg 登录结果消息
 * @param tradingdate 交易日
 * 
 * 登录成功后会：
 * - 保存交易日
 * - 开始查询持仓（然后依次查询账户、成交、订单）
 * 
 * 登录失败会：
 * - 标记适配器为完成状态
 * - 通知管理器减少活跃计数
 */
void TraderAdapter::onLoginResult(bool bSucc, const char* msg, uint32_t tradingdate)
{
	if(!bSucc)                      // 如果登录失败
	{
		WTSLogger::error("[{}]交易账号登录失败: {}", _id.c_str(), msg);  // 记录错误日志
		_mgr->decAlive();           // 通知管理器减少活跃计数
		_done = true;                // 标记为完成状态
	}
	else                            // 如果登录成功
	{
		_date = tradingdate;         // 保存交易日
		WTSLogger::info("[{}]交易账号登录成功, 当前交易日:{}", _id.c_str(), tradingdate);  // 记录成功日志

		_trader_api->queryPositions();	//查持仓，查询结果会通过onRspPosition回调返回
	}
}
```

#### 登出回调 onLogout
```cpp
/**
 * @brief 登出回调的实现
 * 
 * 交易接口登出时的通知，当前实现为空，不做任何处理。
 */
void TraderAdapter::onLogout()
{
	
}
```

#### 交易错误回调 onTraderError
```cpp
/**
 * @brief 交易错误回调的实现
 * @param err 错误信息
 * @param pData 附加数据指针
 * 
 * 处理交易接口的错误通知，记录错误日志。
 */
void TraderAdapter::onTraderError(WTSError* err, void* pData /* = NULL */)
{
	if(err)                          // 如果错误信息存在
		WTSLogger::error("[{}]交易通道出现错误: {}", _id.c_str(), err->getMessage());  // 记录错误日志
}
```

### ITraderSpi 接口实现—查询结果回调

#### 账户资金查询结果回调 onRspAccount
```cpp
/**
 * @brief 账户资金查询结果回调的实现
 * @param ayAccounts 账户信息数组
 * 
 * 处理账户资金查询结果：
 * - 遍历所有账户信息
 * - 提取账户数据（余额、盈亏、保证金等）
 * - 通过Dumper传递给外部回调函数
 * - 查询完成后，继续查询成交明细
 */
void TraderAdapter::onRspAccount(WTSArray* ayAccounts)
{
	if(ayAccounts && ayAccounts->size() > 0)  // 如果账户数组存在且不为空
	{
		for (std::size_t idx = 0; idx < ayAccounts->size(); idx++)  // 遍历所有账户信息
		{
			WTSAccountInfo* accInfo = (WTSAccountInfo*)ayAccounts->at(idx);  // 获取第idx个账户信息对象

			// 通过Dumper转发账户数据到外部回调函数
			// 参数说明：通道ID、交易日、货币、上日余额、当前余额、动态权益、平仓盈亏、浮动盈亏、手续费、保证金、入金、出金、是否最后一条
			getDumper().on_account(_id.c_str(), _date, accInfo->getCurrency(), accInfo->getPreBalance(), accInfo->getBalance(), accInfo->getBalance() + accInfo->getDynProfit(),
				accInfo->getCloseProfit(), accInfo->getDynProfit(), accInfo->getCommission(), accInfo->getMargin(), accInfo->getDeposit(), accInfo->getWithdraw(), idx == ayAccounts->size()-1);
		}
	}

	WTSLogger::info("[{}]资金数据已更新", _id.c_str());  // 记录日志

	if(!_done)                      // 如果还未完成（未完成所有数据查询）
		_trader_api->queryTrades();  // 继续查询成交明细，查询结果会通过onRspTrades回调返回
}
```

#### 持仓查询结果回调 onRspPosition
```cpp
/**
 * @brief 持仓查询结果回调的实现
 * @param ayPositions 持仓信息数组
 * 
 * 处理持仓查询结果：
 * - 遍历所有持仓信息
 * - 提取持仓数据（数量、成本、盈亏等）
 * - 通过Dumper传递给外部回调函数
 * - 查询完成后，继续查询账户资金
 */
void TraderAdapter::onRspPosition(const WTSArray* ayPositions)
{
	if (ayPositions && ayPositions->size() > 0)  // 如果持仓数组存在且不为空
	{
		for (std::size_t idx = 0; idx < ((WTSArray*)ayPositions)->size(); idx++)  // 遍历所有持仓信息
		{
			WTSPositionItem* pItem = (WTSPositionItem*)(((WTSArray*)ayPositions)->at(idx));  // 获取第idx个持仓信息对象
			WTSContractInfo* cInfo = _bd_mgr->getContract(pItem->getCode());  // 从基础数据管理器获取合约信息（只需要合约代码）
			if (cInfo == NULL)      // 如果合约信息不存在，跳过该持仓
				continue;
			WTSCommodityInfo* commInfo = cInfo->getCommInfo();  // 获取品种信息，用于获取数量乘数

			// 通过Dumper转发持仓数据到外部回调函数
			// 参数说明：通道ID、交易所、合约代码、交易日、方向（0=多头，1=空头）、持仓数量、持仓成本、占用保证金、持仓均价、浮动盈亏、数量乘数、是否最后一条
			getDumper().on_position(_id.c_str(), cInfo->getExchg(), cInfo->getCode(), _date, (pItem->getDirection() == WDT_LONG ? 0 : 1),
				pItem->getTotalPosition(), pItem->getPositionCost(), pItem->getMargin(), pItem->getAvgPrice(),
				pItem->getDynProfit(), commInfo->getVolScale(), idx == ayPositions->size() - 1);
		}
	}

	WTSLogger::info("[{}]持仓数据已更新", _id.c_str());  // 记录日志

	if (!_done)                      // 如果还未完成（未完成所有数据查询）
		_trader_api->queryAccount();  // 继续查询账户资金，查询结果会通过onRspAccount回调返回
}
```

#### 成交查询结果回调 onRspTrades
```cpp
/**
 * @brief 成交查询结果回调的实现
 * @param ayTrades 成交信息数组
 * 
 * 处理成交查询结果：
 * - 遍历所有成交记录
 * - 提取成交数据（价格、数量、方向等）
 * - 通过Dumper传递给外部回调函数
 * - 查询完成后，继续查询订单明细
 */
void TraderAdapter::onRspTrades(const WTSArray* ayTrades)
{
	if (ayTrades && ayTrades->size() > 0)  // 如果成交数组存在且不为空
	{
		for (std::size_t idx = 0; idx < ayTrades->size(); idx++)  // 遍历所有成交记录
		{
			WTSTradeInfo* pItem = (WTSTradeInfo*)((WTSArray*)ayTrades)->at(idx);  // 获取第idx个成交信息对象
			WTSContractInfo* cInfo = _bd_mgr->getContract(pItem->getCode(), pItem->getExchg());  // 从基础数据管理器获取合约信息
			if (cInfo == NULL)      // 如果合约信息不存在，跳过该成交
				continue;

			// 通过Dumper转发成交数据到外部回调函数
			// 参数说明：通道ID、交易所、合约代码、交易日、成交编号、订单号、方向、开平、数量、价格、金额、订单类型、成交类型、成交时间、是否最后一条
			getDumper().on_trade(_id.c_str(), cInfo->getExchg(), cInfo->getCode(), _date, pItem->getTradeID(), pItem->getRefOrder(),
				(uint32_t)pItem->getDirection(), (uint32_t)pItem->getOffsetType(), pItem->getVolume(), pItem->getPrice(), pItem->getAmount(),
				(uint32_t)pItem->getOrderType(), (uint32_t)pItem->getTradeType(), pItem->getTradeTime(), idx == ayTrades->size()-1);
		}
	}

	WTSLogger::info("[{}]成交明细已更新", _id.c_str());  // 记录日志

	_trader_api->queryOrders();     // 继续查询订单明细，查询结果会通过onRspOrders回调返回
}
```

#### 订单查询结果回调 onRspOrders
```cpp
/**
 * @brief 订单查询结果回调的实现
 * @param ayOrders 订单信息数组
 * 
 * 处理订单查询结果：
 * - 遍历所有订单记录
 * - 提取订单数据（状态、数量、价格等）
 * - 通过Dumper传递给外部回调函数
 * - 查询完成后，标记适配器为完成状态，通知管理器
 */
void TraderAdapter::onRspOrders(const WTSArray* ayOrders)
{
	if (ayOrders && ayOrders->size() > 0)  // 如果订单数组存在且不为空
	{
		for (std::size_t idx = 0; idx < ayOrders->size(); idx++)  // 遍历所有订单记录
		{
			WTSOrderInfo* pItem = (WTSOrderInfo*)((WTSArray*)ayOrders)->at(idx);  // 获取第idx个订单信息对象
			WTSContractInfo* cInfo = _bd_mgr->getContract(pItem->getCode(), pItem->getExchg());  // 从基础数据管理器获取合约信息
			if (cInfo == NULL)      // 如果合约信息不存在，跳过该订单
				continue;

			// 通过Dumper转发订单数据到外部回调函数
			// 参数说明：通道ID、交易所、合约代码、交易日、订单号、方向、开平、委托数量、剩余数量、已成交数量、价格、订单类型、价格类型、委托时间、订单状态、状态消息、是否最后一条
			getDumper().on_order(_id.c_str(), cInfo->getExchg(), cInfo->getCode(), _date, pItem->getOrderID(), pItem->getDirection(),
				pItem->getOffsetType(), pItem->getVolume(), pItem->getVolLeft(), pItem->getVolTraded(), pItem->getPrice(),
				pItem->getOrderType(), pItem->getPriceType(), pItem->getOrderTime(), pItem->getOrderState(), pItem->getStateMsg(), idx == ayOrders->size()-1);
		}
	}

	WTSLogger::info("[{}]订单明细已更新", _id.c_str());  // 记录日志
	_mgr->decAlive();               // 通知管理器减少活跃计数（所有数据查询完成）
	_done = true;                    // 标记为完成状态
}
```

### ITraderSpi 接口实现—实时推送回调

#### 成交推送回调 onPushTrade
```cpp
/**
 * @brief 成交推送回调的实现
 * @param tradeRecord 成交记录
 * 
 * 处理实时成交推送：
 * - 提取成交数据
 * - 通过Dumper传递给外部回调函数
 * - 与查询结果不同，推送的isLast始终为true
 */
void TraderAdapter::onPushTrade(WTSTradeInfo* tInfo)
{
	WTSContractInfo* cInfo = _bd_mgr->getContract(tInfo->getCode(), tInfo->getExchg());  // 从基础数据管理器获取合约信息
	if (cInfo == NULL)              // 如果合约信息不存在，忽略该成交
		return;

	// 通过Dumper转发成交数据到外部回调函数
	// 参数说明：通道ID、交易所、合约代码、交易日、成交编号、订单号、方向、开平、数量、价格、金额、订单类型、成交类型、成交时间、是否最后一条（推送始终为true）
	getDumper().on_trade(_id.c_str(), cInfo->getExchg(), cInfo->getCode(), _date, tInfo->getTradeID(), tInfo->getRefOrder(),
		(uint32_t)tInfo->getDirection(), (uint32_t)tInfo->getOffsetType(), tInfo->getVolume(), tInfo->getPrice(), tInfo->getAmount(),
		(uint32_t)tInfo->getOrderType(), (uint32_t)tInfo->getTradeType(), tInfo->getTradeTime(), true);
}
```

#### 订单推送回调 onPushOrder
```cpp
/**
 * @brief 订单推送回调的实现
 * @param oInfo 订单信息
 * 
 * 处理实时订单推送：
 * - 如果订单已结束（非活跃状态），刷新账户和持仓
 * - 提取订单数据
 * - 通过Dumper传递给外部回调函数
 */
void TraderAdapter::onPushOrder(WTSOrderInfo* oInfo)
{
	WTSContractInfo* cInfo = _bd_mgr->getContract(oInfo->getCode(), oInfo->getExchg());  // 从基础数据管理器获取合约信息
	if (cInfo == NULL)              // 如果合约信息不存在，忽略该订单
		return;

	//如果订单回报中，订单状态是已结束，则刷新资金和持仓
	if (!oInfo->isAlive())           // 如果订单已结束（已成交、已撤销、已拒绝等）
	{
		_trader_api->queryAccount();  // 刷新账户资金
		_trader_api->queryPositions();  // 刷新持仓信息
	}

	// 通过Dumper转发订单数据到外部回调函数
	// 参数说明：通道ID、交易所、合约代码、交易日、订单号、方向、开平、委托数量、剩余数量、已成交数量、价格、订单类型、价格类型、委托时间、订单状态、状态消息、是否最后一条（推送始终为true）
	getDumper().on_order(_id.c_str(), cInfo->getExchg(), cInfo->getCode(), _date, oInfo->getOrderID(), oInfo->getDirection(),
		oInfo->getOffsetType(), oInfo->getVolume(), oInfo->getVolLeft(), oInfo->getVolTraded(), oInfo->getPrice(),
		oInfo->getOrderType(), oInfo->getPriceType(), oInfo->getOrderTime(), oInfo->getOrderState(), oInfo->getStateMsg(), true);
}
```

### ITraderSpi 接口实现—辅助方法

#### 获取基础数据管理器 getBaseDataMgr
```cpp
/**
 * @brief 获取基础数据管理器的实现
 * @return 返回基础数据管理器指针
 * 
 * 返回初始化时传入的基础数据管理器，供交易接口使用。
 */
IBaseDataMgr* TraderAdapter::getBaseDataMgr()
{
	return _bd_mgr;                  // 返回基础数据管理器指针
}
```

#### 处理交易日志 handleTraderLog
```cpp
/**
 * @brief 处理交易日志的实现
 * @param ll 日志级别
 * @param message 日志消息
 * 
 * 将交易接口的日志转发到WonderTrader日志系统。
 */
void TraderAdapter::handleTraderLog(WTSLogLevel ll, const char* message)
{
	WTSLogger::log_raw(ll, message);  // 直接转发日志到WonderTrader日志系统
}
```

## 交易适配器管理器类 TraderAdapterMgr
```cpp
class TraderAdapterMgr : private boost::noncopyable
```

### 成员
- `TraderAdapterMap _adapters`：适配器映射表，key为通道ID，value为适配器智能指针
  - typedef std::shared_ptr<`TraderAdapter`> TraderAdapterPtr：交易适配器智能指针类型定义
  - typedef std::unordered_map<std::string, `TraderAdapterPtr`>	TraderAdapterMap：交易适配器映射表类型定义，key为通道ID，value为适配器指针
- `std::mutex _mutex`：互斥锁，保护共享数据的并发访问
- `std::atomic<uint32_t> _live_cnt`：原子计数器：活跃适配器数量，线程安全

### 释放所有适配器资源 release
```cpp
/**
 * @brief 释放所有适配器资源的实现
 * 遍历所有适配器，调用其release方法释放资源，然后清空适配器映射表。
 */
void TraderAdapterMgr::release()
{
	for (auto it = _adapters.begin(); it != _adapters.end(); it++)
	{
		it->second->release();
	}
	_adapters.clear();
}
```

### 启动所有适配器 run
```cpp
/**
 * @brief 启动所有适配器的实现
 * 
 * 启动流程：
 * - 设置活跃计数为适配器总数
 * - 遍历所有适配器，调用run方法启动
 * - 记录启动日志
 */
void TraderAdapterMgr::run()
{
	_live_cnt = _adapters.size();
	for (auto it = _adapters.begin(); it != _adapters.end(); it++)
	{
		it->second->run();
	}
	WTSLogger::info("{}个交易通道已启动", _adapters.size());
}
```

### 获取所有适配器的映射表 getAdapters
```cpp
/**
 * @brief 获取所有适配器的映射表
 * @return 返回适配器映射表的常量引用
 * 
 * 用于外部访问适配器映射表，只读访问。
 */
const TraderAdapterMap& getAdapters() const { return _adapters; }
```

### 根据通道ID获取适配器 getAdapter
```cpp
/**
 * @brief 根据通道ID获取适配器的实现
 * @param tname 交易通道ID
 * @return 返回适配器智能指针
 * 
 * 在适配器映射表中查找指定ID的适配器。
 */
TraderAdapterPtr TraderAdapterMgr::getAdapter(const char* tname)
{
	auto it = _adapters.find(tname);
	if (it != _adapters.end())
	{
		return it->second;
	}
	return TraderAdapterPtr();
}
```

### 添加适配器到管理器 addAdapter
```cpp
/**
 * @brief 添加适配器到管理器的实现
 * @param tname 交易通道ID
 * @param adapter 适配器智能指针引用
 * @return 返回添加是否成功
 * 
 * 添加适配器到映射表，如果ID已存在则添加失败。
 */
bool TraderAdapterMgr::addAdapter(const char* tname, TraderAdapterPtr& adapter)
{
	if (adapter == NULL || strlen(tname) == 0)
		return false;

	auto it = _adapters.find(tname);
	if(it != _adapters.end())
	{
		WTSLogger::error("交易通道名称相同: {}", tname);
		return false;
	}
	_adapters[tname] = adapter;
	return true;
}
```

### 检查是否有活跃的适配器 isAnyAlive
```cpp
/**
 * @brief 检查是否有活跃的适配器
 * @return 返回是否有活跃适配器（布尔值）
 * 
 * 检查活跃计数器是否大于0，用于判断是否还有适配器在工作。
 */
bool	isAnyAlive() const {
    return _live_cnt != 0;
}
```

### 获取适配器数量 size
```cpp
/**
 * @brief 获取适配器数量
 * @return 返回适配器映射表的大小（size_t类型）
 */
std::size_t size() const { return _adapters.size(); }
```

### 减少活跃适配器计数 decAlive
```cpp
/**
 * @brief 减少活跃适配器计数的实现
 * 
 * 当一个适配器完成数据查询后，调用此方法减少活跃计数。
 * 使用互斥锁保护计数器的原子操作。
 * 当剩余活跃数较少时，会记录日志。
 */
void TraderAdapterMgr::decAlive()
{
	_mutex.lock(); // 加锁保护共享数据
	auto left = _live_cnt.fetch_sub(1); // 原子操作：减少活跃计数并返回旧值
	_mutex.unlock();
	if (left > 0) // 如果旧值大于0
		left--; // 计算新的剩余数（因为已经减1了）
	if(left <= 2 && left > 0) // 如果剩余活跃数小于等于2且大于0
	{
		for (auto it = _adapters.begin(); it != _adapters.end(); it++)
		{
			TraderAdapterPtr trader = it->second;
			if (!trader->isDone()) // 如果适配器还未完成
				WTSLogger::info("{} is still undone", trader->id());
		}
	}

	WTSLogger::info("{}/{}", left, _adapters.size());
}
```

### 刷新所有已完成的适配器 refresh
```cpp
/**
 * @brief 刷新所有已完成的适配器的实现
 * 
 * 遍历所有适配器，对于已完成的适配器，重新查询持仓和资金。
 * 用于定时刷新功能，保持数据最新。
 */
void TraderAdapterMgr::refresh()
{
	for (auto it = _adapters.begin(); it != _adapters.end(); it++) // 遍历所有适配器
	{
		TraderAdapterPtr trader = it->second; // 获取适配器指针
		if (!trader->isDone()) // 如果适配器还未完成，跳过（只刷新已完成的）
			continue;
		trader->queryPosition(); // 重新查询持仓
		trader->queryFund(); // 重新查询资金
	}
}
```

# 辅助工具类 WtHelper.h/cpp

## 成员
- `static std::string _bin_dir`：静态成员变量：存储模块目录路径，全局唯一

## 获取当前工作目录 get_cwd

## 获取模块目录路径 get_module_dir

## 设置模块目录路径 set_module_dir